# 02 — NLP evaluation

This notebook selects and evaluates the local phishing-email classifier.

**Before running.** Complete Notebook 01 so that the frozen split manifest and both detector representations are available in `artifacts/data/`. This stage requires neither network access nor an API key.

**Workflow.** The evaluation tunes twelve fixed TF–IDF logistic-regression candidates independently on Detector Input v1.0 and v2.0 using the train and validation splits. Representation selection is based on validation balanced accuracy, followed by F1, with the simpler v1.0 preferred if both criteria are tied. After selection, only the chosen configuration is refitted on the combined train and validation data and evaluated once on the held-out test set.

**Outputs.** The workflow saves the validation results, final test predictions, metrics and ranked feature coefficients to `artifacts/nlp/`. Following the dissertation, the notebook presents the five core metrics first and reports ROC AUC and average precision separately as supplementary ranking diagnostics. Accuracy and specificity remain available in `summary.json` but are not repeated here: on the balanced test set, accuracy equals balanced accuracy, while specificity equals one minus the false-positive rate.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

from phishing_detection import (
    CORE_CLASSIFICATION_METRICS,
    SUPPLEMENTARY_RANKING_METRICS,
    StudyConfig,
    run_nlp_study,
)

config = StudyConfig(root=Path.cwd())
nlp_result = run_nlp_study(config)

In [ ]:
core_metric_columns = list(CORE_CLASSIFICATION_METRICS)
supplementary_metric_columns = list(SUPPLEMENTARY_RANKING_METRICS)
validation = pd.DataFrame([{'representation': run['representation'], **run['metrics']} for run in nlp_result['validation_runs']])
print('Core validation metrics (selection uses balanced accuracy, then F1):')
display(validation[['representation', *core_metric_columns]].round(6))
print('Supplementary validation ranking metrics (not used for selection):')
display(validation[['representation', *supplementary_metric_columns]].rename(columns={'pr_auc': 'average_precision'}).round(6))

In [ ]:
test_result = pd.Series(nlp_result['test_metrics'], name='score')
print('Selected representation:', nlp_result['selected_representation'])
print('Held-out test samples:', int(test_result['n']))
print('Five core test metrics:')
display(test_result.loc[core_metric_columns].to_frame())
print('Supplementary cross-threshold ranking metrics:')
display(test_result.loc[supplementary_metric_columns].rename(index={'pr_auc': 'average_precision'}).to_frame())
print('Confusion matrix counts:')
display(pd.Series(nlp_result['test_metrics']['confusion_matrix'], name='count').loc[['tn', 'fp', 'fn', 'tp']].to_frame())

In [ ]:
higher_is_better_metric_columns = [metric for metric in core_metric_columns if metric != 'false_positive_rate']
ax = validation.set_index('representation')[higher_is_better_metric_columns].T.plot(kind='bar', figsize=(9, 5))
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=-13, fontsize=8)
ax.set_title('Validation performance by detector representation')
ax.set_ylabel('Score (truncated axis: 0.95–1.00)')
ax.set_ylim(0.95, 1.00)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Highest-weight TF–IDF features

This section ranks the coefficients from the final model fitted on the combined train and validation data. Negative coefficients indicate associations with legitimate email, whereas positive coefficients indicate associations with phishing. These weights describe predictive associations rather than causal effects. Corpus years, source-specific terms and structural placeholders are deliberately retained so that possible dataset artefacts remain visible.

In [ ]:
features = json.loads((config.nlp_dir / 'top-features.json').read_text(encoding='utf-8'))
limit = 15
feature_table = pd.DataFrame({
    'legitimate_feature': [row['feature'] for row in features['legitimate'][:limit]],
    'legitimate_coefficient': [row['coefficient'] for row in features['legitimate'][:limit]],
    'phishing_feature': [row['feature'] for row in features['phishing'][:limit]],
    'phishing_coefficient': [row['coefficient'] for row in features['phishing'][:limit]],
}, index=range(1, limit + 1))
feature_table.index.name = 'rank'
display(feature_table.round({'legitimate_coefficient': 2, 'phishing_coefficient': 2}))

plot_rows = [
    {'feature': row['feature'], 'coefficient': row['coefficient'], 'class': label}
    for label in ('legitimate', 'phishing')
    for row in features[label][:limit]
]
plot_rows = sorted(plot_rows, key=lambda row: row['coefficient'])
fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(
    [row['feature'] for row in plot_rows],
    [row['coefficient'] for row in plot_rows],
    color=['#4C78A8' if row['class'] == 'legitimate' else '#E45756' for row in plot_rows],
)
ax.axvline(0, color='black', linewidth=0.8)
for bar, row in zip(bars, plot_rows, strict=True):
    value = row['coefficient']
    ax.text(value, bar.get_y() + bar.get_height() / 2, f' {value:.2f} ' if value >= 0 else f' {value:.2f} ', ha='left' if value >= 0 else 'right', va='center', fontsize=8)
ax.set_title('Highest-weight features in the final TF–IDF logistic-regression model')
ax.set_xlabel('Logistic-regression coefficient (legitimate ← 0 → phishing)')
plt.tight_layout()
plt.show()